<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="cognitiveclass.ai logo">
</center>


# **Recurrent Neural Networks**


A recurrent neural network (RNN) is a type of artificial neural network which uses sequential data or time series data as input. Its typically used for ordinal or temporal problems like language translation, speech recognition, and time series forecasting. 

In this lab, we will understand the fundamental building blocks of an RNN. We will train a simple binary text classifier on top of an existing pre-trained module that embeds sentences.


## __Table of Contents__

<ol>
    <li><a href="#toc-objectives">Objectives</a></li>
    <li>
        <a href="#toc-setup">Setup</a>
        <ol>
            <li><a href="#toc-installing-required-libraries">Installing Required Libraries</a></li>
            <li><a href="#toc-importing-required-libraries">Importing Required Libraries</a></li>
        </ol>
    </li>
    <li><a href="#toc-helper-functions">Helper Functions</a></li>
    <li>
        <a href="#toc-rnn-fundamentals">RNN Fundamentals</a>
        <ol>
            <li><a href="#toc-the-core-idea-memory-hidden-state">The Core Idea: Memory (Hidden State)</a></li>
            <li><a href="#toc-the-vanilla-recurrent-neural-network">The Vanilla Recurrent Neural Network</a></li>
            <li><a href="#toc-unrolling-through-time">Unrolling Through Time</a></li>
            <li><a href="#toc-training-the-rnn">Training the RNN</a></li>
        </ol>
    </li>
    <li><a href="#toc-types-of-rnns">Types of RNNs</a></li>
    <li>
        <a href="#toc-pre-trained-rnns">Pre-trained RNNs</a>
        <ol>
            <li><a href="#toc-manual-textvectorizer-and-embedding-layers">Manual text_vectorizer and embedding layers</a></li>
            <li><a href="#toc-universal-sentence-encoder-use">Universal Sentence Encoder (USE)</a></li>
        </ol>
    </li>
</ol>


<a id="Objectives"></a>

## **Objectives** <a id="toc-objectives"></a>

After completing this lab you will be able to:

 - Describe the fundamental building blocks of RNNs.
 - Implement pre-trained RNNs to solve time-series prediction, and forecasting, and text classification tasks


----


## **Setup** <a id="toc-setup"></a>


For this lab, we will be using the following libraries:

*   [`pandas`](https://pandas.pydata.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for managing the data.
*   [`numpy`](https://numpy.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for mathematical operations.
*   [`sklearn`](https://scikit-learn.org/stable/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for machine learning and machine-learning-pipeline related functions.
*   [`seaborn`](https://seaborn.pydata.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for visualizing the data.
*   [`matplotlib`](https://matplotlib.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for additional plotting tools.


### **Installing Required Libraries** <a id="toc-installing-required-libraries"></a>

The following required libraries are pre-installed in the Skills Network Labs environment. However, if you run these notebook commands in a different Jupyter environment (like Watson Studio or Ananconda), you will need to install these libraries by removing the `#` sign before `!mamba` in the code cell below.


In [15]:
# All Libraries required for this lab are listed below. The libraries pre-installed on Skills Network Labs are commented.
# !mamba install -qy pandas numpy seaborn matplotlib scikit-learn
# Note: If your environment doesn't support "!mamba install", use "!pip install"

The following required libraries are __not__ pre-installed in the Skills Network Labs environment. __You will need to run the following cell__ to install them:


In [16]:
%%capture

# !pip install -q "setuptools<81" tensorflow_hub tf-keras
# !pip install -q tensorflow --upgrade
# !mamba install -qy tqdm

### **Importing Required Libraries** <a id="toc-importing-required-libraries"></a>


In [17]:
import importlib.util
import math
import subprocess
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import skillsnetwork
import tensorflow as tf

if importlib.util.find_spec("pkg_resources") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "setuptools<81"])
if importlib.util.find_spec("tf_keras") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tf-keras"])

import tf_keras
import tensorflow_hub as hub
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import reuters
from tensorflow.keras.layers import (
    Conv1D,
    Dense,
    Dropout,
    Embedding,
    GRU,
    LSTM,
    Masking,
    SimpleRNN,
    TextVectorization,
)
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

%matplotlib inline

def warn(*args, **kwargs):
    pass


print(tf.__version__)

# Disable GPU if desired.
tf.config.set_visible_devices([], "GPU")

# Suppress warnings.
warnings.warn = warn
warnings.filterwarnings('ignore')

sns.set_context('notebook')
sns.set_style('white')
np.random.seed(2024)

2.20.0


## **Helper Functions** <a id="toc-helper-functions"></a>


In [18]:
# function to compute the accuracy, precision, recall and F1 score of a model's predictions.
def calculate_results(y_true, y_pred):
    model_accuracy = accuracy_score(y_true, y_pred)
    model_precision, model_recall, model_f1, _ = (
        precision_recall_fscore_support(y_true, y_pred, average="weighted")
    )
    model_results = {
        "accuracy": model_accuracy,
        "precision": model_precision,
        "recall": model_recall,
        "f1": model_f1,
    }
    return model_results

## **RNN Fundamentals** <a id="toc-rnn-fundamentals"></a>

Recurrent Neural Networks (**RNNs**) are a class of neural networks specifically designed to handle **sequential data**. Unlike traditional neural networks, which assume all inputs are independent of each other, RNNs are built to recognize patterns in sequences where the order of the data matters.

**Common examples of sequential data include:**
- **Text:** Words in a sentence.
- **Time Series:** Stock prices or weather readings over time.
- **Audio:** Sound waves sampled over time.
- **Video:** A sequence of image frames.

### **The Core Idea: Memory (Hidden State)** <a id="toc-the-core-idea-memory-hidden-state"></a>
The defining feature of an RNN is its ability to maintain a **hidden state** ($\mathbf{s}_t$). You can think of the hidden state as the network's "memory." 

As the RNN processes a sequence one element at a time, it updates this memory based on:
1. The **current input** (e.g., the current word in a sentence).
2. The **previous hidden state** (the memory of everything it has seen so far).

This allows the network to use context from the beginning of a sequence to understand an element at the end.

**In Keras**, you can implement RNNs using layers like `SimpleRNN`, `LSTM` (Long Short-Term Memory), and `GRU` (Gated Recurrent Unit).

### **The Vanilla Recurrent Neural Network** <a id="toc-the-vanilla-recurrent-neural-network"></a>

A "Vanilla" RNN is the simplest form of a recurrent network. It uses a single cell that is applied repeatedly to every element of the input sequence.

#### **How it Works: The Math**
At each time step $t$, the RNN performs two main calculations:

1. **Update the Memory (Hidden State):**
   The network combines the current input $\mathbf{x}_t$ and the previous memory $\mathbf{s}_{t-1}$ to create a new memory $\mathbf{s}_t$.
   $$\mathbf{s}_t = \operatorname{tanh}(U\mathbf{x}_t + W\mathbf{s}_{t-1} + \mathbf{b}_s)$$

2. **Generate an Output:**
   The network uses the updated memory to produce an output $\mathbf{y}_t$.
   $$\mathbf{y}_t = V\mathbf{s}_t + \mathbf{b}_y$$

#### **The Role of Tanh**
The $\operatorname{tanh}$ (hyperbolic tangent) function is used as an activation function. Its primary job is to "squash" the values of the hidden state to stay between $-1$ and $1$. This prevents the numbers from growing infinitely large (exploding) as they pass through many time steps.

<img src="https://github.com/DataScienceUB/DeepLearningMaster2019/blob/master/images/TanhReal.gif?raw=1" alt="Tanh Graph" style="width: 300px;">



#### **Terminology Breakdown**
- $\mathbf{x}_t$: The input at the current time step $t$.
- $\mathbf{s}_{t-1}$: The "memory" from the previous step.
- $\mathbf{s}_t$: The updated "memory" for the current step.
- $\mathbf{y}_t$: The output produced at time step $t$.
- $U, W, V$: **Weight matrices**. $U$ is for the input, $W$ is for the memory, and $V$ is for the output.
- $\mathbf{b}_s, \mathbf{b}_y$: **Bias terms** that help the model fit the data better.

### **Unrolling Through Time** <a id="toc-unrolling-through-time"></a>

Because an RNN uses the same cell for every step, we can visualize the process by "unrolling" or "unfolding" the network. Instead of seeing one cell that loops, we see a chain of identical cells.

For a sequence of inputs $(\mathbf{x}_1, \mathbf{x}_2, \ldots, \mathbf{x}_T)$, the process looks like this:
- **Step 1:** $\mathbf{s}_1 = \operatorname{tanh}(U\mathbf{x}_1 + W\mathbf{s}_0 + \mathbf{b}_s)$
- **Step 2:** $\mathbf{s}_2 = \operatorname{tanh}(U\mathbf{x}_2 + W\mathbf{s}_1 + \mathbf{b}_s)$
- ...
- **Step T:** $\mathbf{s}_T = \operatorname{tanh}(U\mathbf{x}_T + W\mathbf{s}_{T-1} + \mathbf{b}_s)$

**Crucial Point: Weight Sharing**
The weights $U, W,$ and $V$ are **exactly the same** at every single time step. The model doesn't learn a different rule for the first word versus the tenth word; it learns one general rule for how to update its memory regardless of the position in the sequence.

### **Training the RNN** <a id="toc-training-the-rnn"></a>

Training an RNN is the process of adjusting the weights ($U, W, V$) to minimize the difference between the model's prediction and the actual target.

#### **The Training Cycle**
1. **Forward Pass:** The RNN processes the entire sequence from $t=1$ to $T$, updating its hidden state and producing outputs.
2. **Loss Calculation:** A loss function (like Cross-Entropy) measures how "wrong" the prediction is compared to the true label.
3. **Backward Pass (BPTT):** The model calculates the gradient of the loss with respect to the weights. Because the weights were used at every time step, the gradient must be propagated backward from the last step all the way to the first. This is called **Backpropagation Through Time (BPTT)**.

#### **The Update Rule**
The optimizer (e.g., Adam or SGD) then updates the parameters $\theta$ (which include $U, W, V$):
$$\theta \leftarrow \theta - \eta \nabla_{\theta}L$$
Where:
- $\eta$ is the **learning rate** (how big of a step we take).
- $\nabla_{\theta}L$ is the **gradient** (the direction of steepest increase in loss).

By moving in the opposite direction of the gradient, the model slowly "learns" the best weights to handle the sequential patterns in the data.

## **Types of RNNs** <a id="toc-types-of-rnns"></a>

Predicting the output, $y_t$, at each time step is not always the case. Different RNN architectures can be used to solve different kinds of problems.


|Type|Input|Output|Example problem
|-|-|-|-
|*many-to-many*|An input sequence|An output sequence|Part of Speech (POS) tagging
|*many-to-one*|An input sequence|Value of output sequence for last timestep|Text classification: positive tweet or negative?
|*one-to-many*|Single value of input sequence|An output sequence| Given an input image, predict sequence data


## **Pre-trained RNNs** <a id="toc-pre-trained-rnns"></a>


In this section, we will be experimenting with existing RNNs. We will use the NLP disaster dataset. The dataset contains a `test.csv` and a `train.csv` each of which have the following information:

* The text of a tweet
* A keyword from that tweet (although this may be blank!)
* The location the tweet was sent from (may also be blank)

Our task is to predict whether a given tweet is about a real disaster or not. If so, predict a 1. If not, predict a 0.


Let us start by downloading and unzipping the dataset.


In [19]:
await skillsnetwork.prepare(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML311-Coursera/labs/Module4/L1/nlp_disaster.zip",
    overwrite=True,
)

  0%|          | 0/3 [00:00<?, ?it/s]

Saved to '.'


Now we will read in the train dataset. Here we use `.sample(frac=1)` so all rows in the training dataset are returned in a random order. We also set a random state to ensure reproducibility of results.


In [20]:
train_df = pd.read_csv("train.csv")
# shuffle the dataset
train_df_shuffled = train_df.sample(frac=1, random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


We will use 90% of the entire labelled dataset for training, and 10% of it for testing purposes.


In [21]:
# split the data into 90% training and 10% testing
X_train, X_test, y_train, y_test = train_test_split(
    train_df_shuffled["text"].to_numpy(),
    train_df_shuffled["target"].to_numpy(),
    test_size=0.1,
    random_state=42,
)
X_train.shape, y_train.shape

((6851,), (6851,))

In [22]:
X_train[0:5]

array(['@mogacola @zamtriossu i screamed after hitting tweet',
       'Imagine getting flattened by Kurt Zouma',
       '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
       "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
       'Somehow find you and I collide http://t.co/Ee8RpOahPk'],
      dtype=object)

### **Manual text_vectorizer and embedding layers** <a id="toc-manual-textvectorizer-and-embedding-layers"></a>
Since neural networks cannot process raw text, we need to convert our tweets into a numerical format. 

The `TextVectorization` layer is a preprocessing tool that transforms raw strings into sequences of integers. It performs several steps:
1. **Standardization**: Using `lower_and_strip_punctuation`, it converts all text to lowercase and removes punctuation to ensure that "Apple" and "apple!" are treated as the same word.
2. **Tokenization**: It splits the text into individual words (tokens) based on whitespace.
3. **Integer Mapping**: It assigns a unique integer ID to each unique word in the dataset.

The resulting integer sequences can then be passed into an `Embedding` layer, which converts these IDs into dense vectors that capture the meaning of the words.

In [ ]:
text_vectorizer = TextVectorization(
    max_tokens=None,
    # remove punctuation and make letters lowercase
    standardize="lower_and_strip_punctuation",
    # whitespace delimiter
    split="whitespace",
    # dont group anything, every token alone
    ngrams=None,
    output_mode="int",
    # length of each sentence == length of largest sentence
    output_sequence_length=None,
)

In [24]:
# define hyperparameters

# number of words in the vocabulary
max_vocab_length = 10000
# tweet average length
max_length = 15

Below we define an `Embedding` layer with a vocabulary of 10,000, a vector space of 128 dimensions in which words will be embedded, and input documents that have 15 words each.


In [25]:
embedding = layers.Embedding(
    input_dim=max_vocab_length, output_dim=128, input_length=max_length
)

### **Universal Sentence Encoder (USE)** <a id="toc-universal-sentence-encoder-use"></a>

In the previous section, we built the text processing steps ourselves. We used `TextVectorization` to convert words into integer IDs, and then used an `Embedding` layer to convert those IDs into vectors.

The Universal Sentence Encoder (USE) does this work for us with a pre-trained model. It takes raw text as input and converts each sentence into one fixed-size vector. That vector can then be passed directly into a `Dense` layer for classification.

Because USE already includes its own text processing and embedding logic, we do not combine it with the manual `TextVectorization` and `Embedding` layers. The manual pipeline expects integer token IDs before the `Embedding` layer. USE expects raw strings instead. If we send integer IDs into USE, the input type is wrong. If we send USE vectors into an `Embedding` layer, the input type is also wrong because an `Embedding` layer expects integer IDs, not sentence vectors.

So we use one approach or the other:
- Manual approach: raw text -> `TextVectorization` -> `Embedding` -> classifier
- USE approach: raw text -> Universal Sentence Encoder -> classifier

The `hub.KerasLayer` lets us use the pre-trained USE model as a normal Keras layer. The Universal Sentence Encoder was trained on a large amount of text, so it can create useful sentence vectors for tasks such as text classification, semantic similarity, and clustering.

> We can train a simple binary text classifier on top of any TF-Hub module that can embed sentences. The Universal Sentence Encoder was partially trained with custom text classification tasks in mind. These kinds of classifiers can be trained to perform a wide variety of classification tasks often with a very small amount of labeled examples.

More on this is found in the Tensorflow Hub [documentation](https://tfhub.dev/google/universal-sentence-encoder/4)


In [26]:
encoder_layer = hub.KerasLayer(
    "https://tfhub.dev/google/universal-sentence-encoder/4",
    input_shape=[],
    dtype=tf.string,
    trainable=False,
    name="pretrained",
)

The `encoder_layer` will take as input variable length English text and the output is a 512 dimensional vector.


We will add a Dense layer with unit 1 to create a simple binary text classifier on top of the TF-Hub module. Because `hub.KerasLayer` uses the Keras 2 compatibility API, this model is built with `tf_keras` instead of the newer Keras 3 API exposed through `tf.keras`.


In [27]:
model = tf_keras.Sequential(
    [
        encoder_layer,
        tf_keras.layers.Dense(1, activation="sigmoid"),
    ],
    name="model_pretrained",
)
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

model.fit(
    x=X_train,
    y=y_train,
    epochs=20,
    validation_data=(X_test, y_test),
)

Epoch 1/20
215/215 [==============================] - 1s 2ms/step - loss: 0.6552 - accuracy: 0.7068 - val_loss: 0.6177 - val_accuracy: 0.7808
Epoch 2/20
215/215 [==============================] - 0s 2ms/step - loss: 0.5858 - accuracy: 0.7884 - val_loss: 0.5666 - val_accuracy: 0.7913
Epoch 3/20
215/215 [==============================] - 0s 2ms/step - loss: 0.5413 - accuracy: 0.7958 - val_loss: 0.5336 - val_accuracy: 0.7953
Epoch 4/20
215/215 [==============================] - 0s 2ms/step - loss: 0.5117 - accuracy: 0.7973 - val_loss: 0.5118 - val_accuracy: 0.7966
Epoch 5/20
215/215 [==============================] - 0s 2ms/step - loss: 0.4911 - accuracy: 0.7993 - val_loss: 0.4967 - val_accuracy: 0.7992
Epoch 6/20
215/215 [==============================] - 0s 2ms/step - loss: 0.4760 - accuracy: 0.8013 - val_loss: 0.4859 - val_accuracy: 0.7979
Epoch 7/20
215/215 [==============================] - 0s 2ms/step - loss: 0.4646 - accuracy: 0.8035 - val_loss: 0.4783 - val_accuracy: 0.8018
Epoch 

Epoch 1/20
215/215 [==============================] - 1s 2ms/step - loss: 0.6552 - accuracy: 0.7068 - val_loss: 0.6177 - val_accuracy: 0.7808
Epoch 2/20
215/215 [==============================] - 0s 2ms/step - loss: 0.5858 - accuracy: 0.7884 - val_loss: 0.5666 - val_accuracy: 0.7913
Epoch 3/20
215/215 [==============================] - 0s 2ms/step - loss: 0.5413 - accuracy: 0.7958 - val_loss: 0.5336 - val_accuracy: 0.7953
Epoch 4/20
215/215 [==============================] - 0s 2ms/step - loss: 0.5117 - accuracy: 0.7973 - val_loss: 0.5118 - val_accuracy: 0.7966
Epoch 5/20
215/215 [==============================] - 0s 2ms/step - loss: 0.4911 - accuracy: 0.7993 - val_loss: 0.4967 - val_accuracy: 0.7992
Epoch 6/20
215/215 [==============================] - 0s 2ms/step - loss: 0.4760 - accuracy: 0.8013 - val_loss: 0.4859 - val_accuracy: 0.7979
Epoch 7/20
215/215 [==============================] - 0s 2ms/step - loss: 0.4646 - accuracy: 0.8035 - val_loss: 0.4783 - val_accuracy: 0.8018
Epoch 

The following cell checks how well the model performs on the test data. `model.predict(X_test)` gives a probability for each tweet. Since this is a binary classification problem, `tf.round(...)` changes each probability into a class label: `0` or `1`. The model usually returns predictions with an extra dimension, such as `(num_examples, 1)`. `tf.squeeze(...)` removes that unnecessary dimension and changes the shape to `(num_examples,)`. This makes the predictions the same shape as `y_test`, so `calculate_results` can compare them correctly.


In [28]:
calculate_results(
    y_true=y_test, y_pred=tf.squeeze(tf.round(model.predict(X_test)))
)

24/24 [==============================] - 0s 2ms/step


24/24 [==============================] - 0s 2ms/step


{'accuracy': 0.8018372703412073,
 'precision': 0.8024564540108415,
 'recall': 0.8018372703412073,
 'f1': 0.800794135155016}

The model is able to predict the tweet class with a fairly high accuracy.


## **Authors**


[Kopal Garg](https://www.linkedin.com/in/gargkopal/)


Kopal is a Masters student in Computer Science at the University of Toronto.


Su Wu

## **Change Log**


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2022-07-18|0.1|Kopal|Create Lab|
|2022-08-30|0.1|Steve Hord|QA pass edits|
| 2026-05-24     | 0.2     | Su Wu| Documentation and script edits    |


Copyright © 2022 IBM Corporation. All rights reserved.
